In [1]:
import sys
sys.path.insert(0,'..')
from warnings import filterwarnings
filterwarnings("ignore")
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import os
import random
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from apex import amp
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from source.version7.data import trainLoader
from source.version7.model import EfficientModel
from source.version7.train import trainModel
from source.version7.loss import BCELoss
from catalyst.data.sampler import BalanceClassSampler

In [3]:
SEED = 42

def seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    return None

seed(42)

In [ ]:
def train(fold):
    loader = {}
    loader['image_path'] = '../../data/cdeotte/train/train/'
    loader['label_path'] = '../../data/cdeotte/data_extra.csv'
    loader['fold_idx'] = fold
    train, valid = trainLoader(**loader)
    params = {}
    params['batch_size'] = 10
    params['num_workers'] = 4
    params['drop_last'] = True
    train = DataLoader(train, shuffle=True, **params)
    valid = DataLoader(valid, **params)
    model = EfficientModel()
    model = model.to('cuda:0')
    optimizer = AdamW(model.parameters(), lr=1e-05, weight_decay=0.)
    schedular = ReduceLROnPlateau(optimizer, factor=0.25, patience=0, min_lr=1e-8)
    model, optimizer = amp.initialize(model, optimizer, opt_level='O2', verbosity=False)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train
    trainer['valid_data'] = valid
    trainer['loss_fn'] = BCELoss()
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../model/version7/model_{}.pt'.format(fold)
    trainer['epochs'] = 20
    trainer['batch'] = 10
    trainer['scheduler'] = schedular
    trainModel(**trainer)
    model.cpu()
    del model
    return None

In [ ]:
train(0)

Train Images: 35499 Valid Images: 6527
Loaded pretrained weights for efficientnet-b5


100% 35490/35490 [29:54<00:00, 19.78it/s, trn_ls=0.8048, val_ls=0.3877, val_mt=0.8646]
100% 35490/35490 [31:15<00:00, 18.92it/s, trn_ls=0.6023, val_ls=0.3328, val_mt=0.8845]
100% 35490/35490 [30:18<00:00, 19.52it/s, trn_ls=0.5430, val_ls=0.3208, val_mt=0.8915]
 26% 9280/35490 [07:22<21:15, 20.54it/s, trn_ls=0.53210]

In [ ]:
train(1)

In [ ]:
train(2)

In [ ]:
train(3)

In [ ]:
train(4)